In [1]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from tqdm import tqdm
import torch

/Users/marconatale/Documents/GitHub/Magistrale/MNLP/HW2/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/marconatale/Documents/GitHub/Magistrale/MNLP/HW2/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("mps") if torch.backends.mps.is_available() else \
        torch.device("cuda") if torch.cuda.is_available() else \
        torch.device("cpu")

In [3]:
print(f"Using device: {device}")

Using device: mps


# Loading the NLLB Translation Model

In [4]:
# Define model name
model_name = "facebook/nllb-200-3.3B"

# Load tokenizer and model
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model...")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

Loading tokenizer...
Loading model...


Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00, 14.08it/s]


# Reading the Dataset

We'll load the CSV file containing Old Italian sentences that need to be translated to Modern Italian.

In [5]:
# Read the CSV file
df = pd.read_csv('dataset.csv')

# Display basic information
print(f"Dataset shape: {df.shape}")
print("\nFirst 5 rows:")
df.head()

Dataset shape: (97, 4)

First 5 rows:


,Author,Date,Region,Sentence
0,Brunetto Latini,1260-61,fior.,quella guerra ben fatta l' opera perché etc. E...
1,Bono Giamboni,1292,fior.,"crudele, e di tutte le colpe pigli vendetta, c..."
2,Valerio Massimo (red. V1,1336,fior.,Non d' altra forza d' animo fue ornato Ponzio ...
3,Lucano volg. (ed. Marinoni),1330/40,prat.,Se questo piace a tutti e se 'l tempo hae biso...
4,Brunetto Latini,1260-61,fior.,Officio di questa arte pare che sia dicere app...


# Setting up Translator

In [6]:
source_lang = "ita_Latn"
target_lang = "ita_Latn"
translator = pipeline('translation', model=model, tokenizer=tokenizer, src_lang=source_lang, tgt_lang=target_lang, max_length = 400)

Device set to use mps:0


# Translating the Sentences

We'll now translate all sentences in the dataset from Old Italian to Modern Italian and add them as a new column.

In [7]:
test_subset = df.head(10).copy()
test_translations = []

# Translate each sentence
for sentence in tqdm(test_subset['Sentence']):
    translated = translator(sentence)[0]['translation_text']
    test_translations.append(translated)

# Add translations to the test dataframe
test_subset['Modern_Italian'] = test_translations

# Show the results
print("\nTranslation Results:")
test_subset[['Sentence', 'Modern_Italian']].head(10)

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:23<00:00,  2.31s/it]


Translation Results:


,Sentence,Modern_Italian
0,quella guerra ben fatta l' opera perché etc. E...,E d'altra parte Aiaces era un cavaliere franco...
1,"crudele, e di tutte le colpe pigli vendetta, c...","crudele, e per ogni colpa vendicatevi, come di..."
2,Non d' altra forza d' animo fue ornato Ponzio ...,Non per altra forza d'animo fu decorato Ponzio...
3,Se questo piace a tutti e se 'l tempo hae biso...,Se a tutti piace e se il tempo ha bisogno di P...
4,Officio di questa arte pare che sia dicere app...,L'obiettivo di questa arte sembra essere quell...
5,via alle mortali onde. Ecco e larghi ventipio...,"Ecco, i venti venti larghi scagliano nubi riso..."
6,si nega. Però che or chi spererebbe quello che...,E' vero. Ma se qualcuno spera di avere la stes...
7,"bestiame, i vendimenti de' morti et le presure...","bestiame, vendite di cadaveri e prelievi di vi..."
8,"Acciocché quegli, il quale ora per le sue gran...","E poiché lui, che ora per la sua grande realt ..."
9,controversie adusandosi gli uomini spessamente...,Le controversie portano spesso gli uomini a st...


In [7]:
translations = []

# Translate each sentence
print("Translating sentences...")
for sentence in tqdm(df['Sentence']):
    translated = translator(sentence)[0]['translation_text']
    translations.append(translated)

# Add translations to the dataframe
df['Modern_Italian'] = translations

df[['Sentence', 'Modern_Italian']].head()

Translating sentences...


100%|██████████| 97/97 [02:13<00:00,  1.38s/it]


,Sentence,Modern_Italian
0,quella guerra ben fatta l' opera perché etc. E...,E d'altra parte Aiaces era un cavaliere franco...
1,"crudele, e di tutte le colpe pigli vendetta, c...","crudele, e per ogni colpa vendicatevi, come di..."
2,Non d' altra forza d' animo fue ornato Ponzio ...,Non per altra forza d'animo fu decorato Ponzio...
3,Se questo piace a tutti e se 'l tempo hae biso...,Se a tutti piace e se il tempo ha bisogno di P...
4,Officio di questa arte pare che sia dicere app...,L'obiettivo di questa arte sembra essere quell...


# Saving the Results

We'll save the original sentences along with their translations to a new CSV file.

In [9]:
output_file = 'nllb/nllb_translations.csv'
df.to_csv(output_file, index=False)

print(f"Saved translations to {output_file}")

Saved translations to nllb/nllb_translations.csv


In [9]:
# Read the saved translations file
translations_df = pd.read_csv('nllb/nllb_translations.csv')

# Show the first few rows
print("\nFirst 5 rows:")
translations_df.head()


First 5 rows:


,Author,Date,Region,Sentence,Modern_Italian
0,Brunetto Latini,1260-61,fior.,quella guerra ben fatta l' opera perché etc. E...,E d'altra parte Aiaces era un cavaliere franco...
1,Bono Giamboni,1292,fior.,"crudele, e di tutte le colpe pigli vendetta, c...","crudele, e per ogni colpa vendicatevi, come di..."
2,Valerio Massimo (red. V1,1336,fior.,Non d' altra forza d' animo fue ornato Ponzio ...,Non per altra forza d'animo fu decorato Ponzio...
3,Lucano volg. (ed. Marinoni),1330/40,prat.,Se questo piace a tutti e se 'l tempo hae biso...,Se a tutti piace e se il tempo ha bisogno di P...
4,Brunetto Latini,1260-61,fior.,Officio di questa arte pare che sia dicere app...,L'obiettivo di questa arte sembra essere quell...


In [10]:
import os
from dotenv import load_dotenv
from google import genai
import time

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

def evaluate_translation(row, without_context = False) -> int:
    criteria = (
        "1. Completely unacceptable translation: the translation has no pertinence with the original meaning, the generated sentence is either gibberish or something that makes no sense.\n"
        "2. Severe semantic errors, omissions or substantial add ons on the original sentence. The errors are of semantic and syntactic nature. It’s still something no human would ever write.\n"
        "3. Partially wrong translation, the translation is lackluster, it contains errors, but are mostly minor errors, like typos, or small semantic errors.\n"
        "4. Good translation. The translation is mostly right, substantially faithful to the original text, but the style does not perfectly match the original sentence, still fluent and comprehensible, and could semantically acceptable.\n"
        "5. Perfect translation. The translation is accurate, fluent, complete and coherent. It retained the original meaning as much as it could."
    )

    if without_context:
        # Prompt without context
        prompt = (
            f"Rate the following translation on a scale of 1-5 based on these criteria:\n"
            f"{criteria}\n\n"
            f"Original sentence:\n\"{row['Sentence']}\"\n\n"
            f"Translated sentence:\n\"{row['Modern_Italian']}\"\n\n"
            f"Provide only the rating (1-5)."
        )
    else:
        # Prompt with context
        prompt = (
            f"Rate this translation on a scale of 1-5 based on these criteria:\n"
            f"{criteria}\n\n"
            "Context:\n"
            f" • Author: {row['Author']}\n"
            f" • Date: {row['Date']}\n"
            f" • Region: {row['Region']}\n\n"
            f"Original sentence:\n\"{row['Sentence']}\"\n\n"
            f"Translated into Modern Italian:\n\"{row['Modern_Italian']}\"\n\n"
            "Provide only the rating (1-5)."
        )
    resp = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt
    )
    try:
        return int(resp.text.strip())
    except ValueError:
        return None

In [11]:
RATE_LIMIT_SECONDS = 4.0

ratings = []
for _, row in tqdm(translations_df.iterrows(), total=len(translations_df), desc="Evaluating translations"):
    rating = evaluate_translation(row)
    ratings.append(rating)
    time.sleep(RATE_LIMIT_SECONDS)

translations_df['gemini_eval'] = ratings

Evaluating translations: 100%|██████████| 97/97 [07:06<00:00,  4.39s/it]


In [12]:
# Now evaluate translations without context
RATE_LIMIT_SECONDS = 4.0

ratings_no_context = []
for _, row in tqdm(translations_df.iterrows(), total=len(translations_df), desc="Evaluating without context"):
    rating = evaluate_translation(row, without_context=True)
    ratings_no_context.append(rating)
    time.sleep(RATE_LIMIT_SECONDS)

translations_df['gemini_eval_no_context'] = ratings_no_context

# Print summary statistics
print("\nEvaluation summary:")
print(f"Average rating with context: {translations_df['gemini_eval'].mean():.2f}")
print(f"Average rating without context: {translations_df['gemini_eval_no_context'].mean():.2f}")
print(f"Difference: {(translations_df['gemini_eval'] - translations_df['gemini_eval_no_context']).mean():.2f}")

translations_df.to_csv('nllb/nllb_translations_with_eval.csv', index=False)

Evaluating without context: 100%|██████████| 97/97 [07:09<00:00,  4.43s/it]


Evaluation summary:
Average rating with context: 4.34
Average rating without context: 4.15
Difference: 0.19
